# Credit Risk Classification — Upgraded Pipeline
### Dataset: Home Credit Default Risk (Kaggle)
( https://www.kaggle.com/competitions/home-credit-default-risk/data)


## Step 0: Install dependencies

In [ ]:
# Check for library installs
# !pip install pandas numpy scikit-learn xgboost matplotlib

## Step 1: Load the Kaggle Home Credit dataset

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# CONFIG: change filenames here if yours are named differently
TRAIN_FILE = 'application_train.csv'
TEST_FILE  = 'application_test.csv'
TARGET_COL = 'TARGET'       # bad=1, good=0
ID_COL     = 'SK_ID_CURR'   # account identifier
# ---------------------------------------------------------

raw_df = pd.read_csv(TRAIN_FILE)

print(f"Dataset shape : {raw_df.shape}")
print(f"Bad accounts  : {raw_df[TARGET_COL].sum()} ({raw_df[TARGET_COL].mean()*100:.2f}%)")
print(f"Good accounts : {(raw_df[TARGET_COL]==0).sum()}")
raw_df.head()

## Step 2: Encode categoricals + drop columns with >80% missing

In [ ]:
# Home Credit has some categorical columns — label-encode them
# so the rest of the pipeline (RF, XGB) can handle all features uniformly
from sklearn.preprocessing import LabelEncoder

working_df = raw_df.copy()

cat_cols = working_df.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    working_df[col] = working_df[col].astype(str)
    working_df[col] = le.fit_transform(working_df[col])

print(f"Label-encoded {len(cat_cols)} categorical columns: {cat_cols[:5]} ...")

# Drop columns missing more than 80% of values
missing_cutoff  = 0.8
cols_to_remove  = [c for c in working_df.columns if working_df[c].isnull().mean() > missing_cutoff]
loan_data       = working_df.drop(columns=cols_to_remove)

print(f"Dropped {len(cols_to_remove)} columns (>80% missing): {cols_to_remove}")
print(f"Remaining shape: {loan_data.shape}")

## Step 3: Impute remaining missing values with column mean

In [ ]:
from sklearn.impute import SimpleImputer

mean_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
arr = loan_data.values
mean_imputer = mean_imputer.fit(arr[:, ::])
arr[:, ::]   = mean_imputer.transform(arr[:, ::])
loan_data.iloc[:, ::] = arr[:, ::]

print(f"Missing values remaining: {loan_data.isnull().sum().sum()}")

## Step 4: Hyperparameter search — 50 random trials
Searching over: RF sampling ratio, XGB sampling ratio, number of top features



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from scipy import stats
import random

# ---- helper: compute KS statistic and Gini from predicted probabilities ----
def compute_ks_gini(y_true, y_prob):
    """
    KS Statistic: max separation between cumulative bad and good rate curves.
    Gini Coefficient: 2 * AUC - 1  (ranges 0 to 1; higher is better)
    Both are standard credit risk model evaluation metrics.
    """
    df_eval = pd.DataFrame({'label': y_true, 'prob': y_prob})
    df_eval = df_eval.sort_values('prob', ascending=False).reset_index(drop=True)

    total_bad  = df_eval['label'].sum()
    total_good = len(df_eval) - total_bad

    df_eval['cum_bad_rate']  = df_eval['label'].cumsum() / total_bad
    df_eval['cum_good_rate'] = (1 - df_eval['label']).cumsum() / total_good

    ks_score = (df_eval['cum_bad_rate'] - df_eval['cum_good_rate']).abs().max()

    auc_score   = roc_auc_score(y_true, y_prob)
    gini_score  = 2 * auc_score - 1

    return round(ks_score, 4), round(gini_score, 4)


search_log = []
n_experiments      = 50
rf_sampling_range  = (1, 40)
xgb_sampling_range = (30, 40)
feat_count_range   = (50, 300)

positives = loan_data[loan_data[TARGET_COL] == 1]
negatives = loan_data[loan_data[TARGET_COL] == 0]

for exp in range(n_experiments):
    rf_neg_ratio  = random.uniform(*rf_sampling_range)
    xgb_neg_ratio = random.uniform(*xgb_sampling_range)
    n_top_feats   = random.randint(*feat_count_range)

    # Build RF training set
    neg_sample_rf = negatives.sample(n=int(rf_neg_ratio * len(positives)), random_state=42)
    rf_dataset    = pd.concat([positives, neg_sample_rf])
    feat_cols     = rf_dataset.drop(columns=[TARGET_COL, ID_COL])
    target_col    = rf_dataset[TARGET_COL]

    # RF feature selection
    rf_clf       = RandomForestClassifier(random_state=42, n_jobs=-1)
    rf_clf.fit(feat_cols, target_col)
    ranked_feats = pd.Series(rf_clf.feature_importances_, index=feat_cols.columns).sort_values(ascending=False)
    selected_feats = ranked_feats.head(n_top_feats).index

    # Build XGB training set
    neg_sample_xgb = negatives.sample(n=int(xgb_neg_ratio * len(positives)), random_state=42)
    xgb_dataset    = pd.concat([positives, neg_sample_xgb])
    X_xgb = xgb_dataset[selected_feats]
    y_xgb = xgb_dataset[TARGET_COL]

    # Train XGBoost
    class_balance = len(y_xgb[y_xgb == 0]) / len(y_xgb[y_xgb == 1])
    xgb_clf = XGBClassifier(scale_pos_weight=class_balance, random_state=42, eval_metric='logloss', use_label_encoder=False)
    xgb_clf.fit(X_xgb, y_xgb)

    # Evaluate on full dataset
    X_eval   = loan_data[selected_feats]
    y_eval   = loan_data[TARGET_COL]
    preds    = xgb_clf.predict(X_eval)
    probs    = xgb_clf.predict_proba(X_eval)[:, 1]

    acc          = accuracy_score(y_eval, preds)
    cm           = confusion_matrix(y_eval, preds)
    ks, gini     = compute_ks_gini(y_eval.values, probs)

    tp = cm[1][1]; fp = cm[0][1]; fn = cm[1][0]
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    print(f"Exp {exp+1:02d} | rf={rf_neg_ratio:.1f} xgb={xgb_neg_ratio:.1f} "
          f"feats={n_top_feats} | acc={acc:.4f} KS={ks:.4f} Gini={gini:.4f}")

    search_log.append({
        'experiment': exp + 1,
        'rf_ratio'  : rf_neg_ratio,
        'xgb_ratio' : xgb_neg_ratio,
        'n_features': n_top_feats,
        'accuracy'  : acc,
        'ks_stat'   : ks,
        'gini'      : gini,
        'precision' : prec,
        'recall'    : rec,
        'f1'        : f1
    })

search_results = pd.DataFrame(search_log)

# Pick best config by KS score (more meaningful than accuracy for credit risk)
top_config = search_results.loc[search_results['ks_stat'].idxmax()]
print("\n=== Best Configuration (by KS Score) ===")
print(top_config[['rf_ratio','xgb_ratio','n_features','accuracy','ks_stat','gini']])

## Step 5: Train final model using best hyperparameters

In [ ]:
from sklearn.model_selection import train_test_split

# Pull best config values from search
best_rf_ratio  = top_config['rf_ratio']
best_xgb_ratio = top_config['xgb_ratio']
best_n_feats   = int(top_config['n_features'])

high_risk = loan_data[loan_data[TARGET_COL] == 1]
low_risk  = loan_data[loan_data[TARGET_COL] == 0]

# RF feature selection dataset
low_risk_rf = low_risk.sample(n=int(best_rf_ratio * len(high_risk)), random_state=42)
combined_rf = pd.concat([high_risk, low_risk_rf])
X_rf_full   = combined_rf.drop(columns=[TARGET_COL, ID_COL])
y_rf_full   = combined_rf[TARGET_COL]

rf_selector  = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_selector.fit(X_rf_full, y_rf_full)
feat_ranking = pd.Series(rf_selector.feature_importances_, index=X_rf_full.columns).sort_values(ascending=False)
best_feats   = feat_ranking.head(best_n_feats).index

# XGB training dataset
low_risk_xgb  = low_risk.sample(n=int(best_xgb_ratio * len(high_risk)), random_state=42)
combined_xgb  = pd.concat([high_risk, low_risk_xgb])
X_train_final = combined_xgb[best_feats]
y_train_final = combined_xgb[TARGET_COL]

# Hold-out test split from the full cleaned data
X_all = loan_data[best_feats]
y_all = loan_data[TARGET_COL]
X_train_split, X_test_split, y_train_split, y_test_split = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

# Train final model
pos_weight  = len(y_train_final[y_train_final == 0]) / len(y_train_final[y_train_final == 1])
final_model = XGBClassifier(scale_pos_weight=pos_weight, random_state=42,
                             eval_metric='logloss', use_label_encoder=False)
final_model.fit(X_train_final, y_train_final)

# Get predictions and probabilities
X_full_eval   = loan_data[best_feats]
y_full_eval   = loan_data[TARGET_COL]

train_preds   = final_model.predict(X_train_final)
full_preds    = final_model.predict(X_full_eval)
full_probs    = final_model.predict_proba(X_full_eval)[:, 1]
test_probs    = final_model.predict_proba(X_test_split)[:, 1]

train_acc = accuracy_score(y_train_final, train_preds)
full_acc  = accuracy_score(y_full_eval, full_preds)

print(f"Training Accuracy : {train_acc:.4f}")
print(f"Full Dataset Acc  : {full_acc:.4f}")
print(f"\nConfusion Matrix (Full):\n{confusion_matrix(y_full_eval, full_preds)}")

##KS(Kolmogorov-Smirnov) Statistic & Gini Coefficient


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def plot_ks_curve(y_true, y_prob, title='KS Curve'):
    """
    Plots the cumulative bad-rate and good-rate curves and highlights
    the maximum separation (KS statistic).
    """
    df_ks = pd.DataFrame({'label': y_true, 'prob': y_prob})
    df_ks = df_ks.sort_values('prob', ascending=False).reset_index(drop=True)

    total_bad  = df_ks['label'].sum()
    total_good = len(df_ks) - total_bad

    df_ks['cum_bad_pct']  = df_ks['label'].cumsum() / total_bad
    df_ks['cum_good_pct'] = (1 - df_ks['label']).cumsum() / total_good
    df_ks['pct_population'] = (df_ks.index + 1) / len(df_ks)
    df_ks['separation']  = (df_ks['cum_bad_pct'] - df_ks['cum_good_pct']).abs()

    ks_idx  = df_ks['separation'].idxmax()
    ks_val  = df_ks.loc[ks_idx, 'separation']
    ks_x    = df_ks.loc[ks_idx, 'pct_population']
    ks_bad  = df_ks.loc[ks_idx, 'cum_bad_pct']
    ks_good = df_ks.loc[ks_idx, 'cum_good_pct']

    auc   = roc_auc_score(y_true, y_prob)
    gini  = 2 * auc - 1

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -- KS Curve --
    ax1 = axes[0]
    ax1.plot(df_ks['pct_population'], df_ks['cum_bad_pct'],  color='#D62728', lw=2, label='Cumulative Bad Rate')
    ax1.plot(df_ks['pct_population'], df_ks['cum_good_pct'], color='#1F77B4', lw=2, label='Cumulative Good Rate')
    ax1.vlines(ks_x, ks_good, ks_bad, colors='green', lw=2.5, linestyles='--', label=f'KS = {ks_val:.4f}')
    ax1.scatter([ks_x], [(ks_bad + ks_good) / 2], color='green', zorder=5, s=60)
    ax1.set_xlabel('Population (sorted by predicted probability)')
    ax1.set_ylabel('Cumulative Rate')
    ax1.set_title(f'{title}\nKS = {ks_val:.4f}  |  Gini = {gini:.4f}')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # -- ROC Curve (for Gini) --
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax2 = axes[1]
    ax2.plot(fpr, tpr, color='#9467BD', lw=2, label=f'ROC (AUC={auc:.4f})')
    ax2.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
    ax2.fill_between(fpr, tpr, alpha=0.15, color='#9467BD')
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title(f'ROC Curve\nGini = 2×AUC−1 = {gini:.4f}')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('ks_gini_plot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"\n{'='*40}")
    print(f"  KS Statistic : {ks_val:.4f}  {'GOOD' if ks_val >= 0.4 else 'MODERATE' if ks_val >= 0.2 else 'WEAK'}")
    print(f"  Gini Coeff   : {gini:.4f}  {'GOOD' if gini >= 0.5 else 'MODERATE' if gini >= 0.3 else 'WEAK'}")
    print(f"  AUC-ROC      : {auc:.4f}")
    print(f"{'='*40}")
    return ks_val, gini

print("--- KS & Gini on Full Training Dataset ---")
ks_train, gini_train = plot_ks_curve(y_full_eval.values, full_probs, title='KS Curve — Full Training Data')

print("\n--- KS & Gini on Hold-out Test Split ---")
ks_test, gini_test = plot_ks_curve(y_test_split.values, test_probs, title='KS Curve — Test Split (20%)')


##Threshold Optimization


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_sweep = np.arange(0.05, 0.95, 0.01)
thresh_results  = []

y_true_test = y_test_split.values

for t in threshold_sweep:
    preds_t = (test_probs >= t).astype(int)
    if preds_t.sum() == 0:   # skip if no positives predicted
        continue
    prec_t = precision_score(y_true_test, preds_t, zero_division=0)
    rec_t  = recall_score(y_true_test,    preds_t, zero_division=0)
    f1_t   = f1_score(y_true_test,        preds_t, zero_division=0)
    acc_t  = accuracy_score(y_true_test,  preds_t)
    thresh_results.append({'threshold': t, 'precision': prec_t,
                           'recall': rec_t, 'f1': f1_t, 'accuracy': acc_t})

thresh_df = pd.DataFrame(thresh_results)

# Find optimal threshold
best_thresh_row  = thresh_df.loc[thresh_df['f1'].idxmax()]
optimal_thresh   = best_thresh_row['threshold']

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='#1F77B4', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='#D62728', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1 Score',  color='#2CA02C', lw=2)
ax.axvline(x=optimal_thresh, color='orange', lw=2.5, linestyle='--',
           label=f'Best Threshold = {optimal_thresh:.2f} (F1={best_thresh_row["f1"]:.4f})')
ax.axvline(x=0.5, color='gray', lw=1.5, linestyle=':', label='Default 0.5')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Optimisation — Precision / Recall / F1 vs Threshold')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nDefault threshold (0.50):")
default_row = thresh_df.iloc[(thresh_df['threshold'] - 0.5).abs().argsort()[:1]]
print(default_row[['threshold','precision','recall','f1']].to_string(index=False))

print(f"\nOptimal threshold ({optimal_thresh:.2f}):")
print(best_thresh_row[['threshold','precision','recall','f1']].to_string())

# Apply optimal threshold
optimised_preds = (test_probs >= optimal_thresh).astype(int)
print(f"\nConfusion matrix at optimal threshold ({optimal_thresh:.2f}):")
print(confusion_matrix(y_true_test, optimised_preds))


## NEW FEATURE 3: Calibration Curve


In [ ]:
from sklearn.calibration import calibration_curve

fraction_of_positives, mean_predicted_prob = calibration_curve(
    y_true_test, test_probs, n_bins=15, strategy='quantile'
)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfect Calibration')
ax.plot(mean_predicted_prob, fraction_of_positives,
        's-', color='#9467BD', lw=2, markersize=7, label='XGBoost Model')
ax.fill_between(mean_predicted_prob, fraction_of_positives, mean_predicted_prob,
                alpha=0.15, color='#9467BD')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Actual Fraction of Positives')
ax.set_title('Calibration Curve\n(How trustworthy are the predicted probabilities?)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# Quantify calibration error
expected_calib_error = np.mean(np.abs(fraction_of_positives - mean_predicted_prob))
print(f"Expected Calibration Error (ECE): {expected_calib_error:.4f}")
print("(Closer to 0 = better calibrated)")

##Learning Curve

In [ ]:
from sklearn.model_selection import learning_curve

# Use a lightweight XGB for the learning curve (faster)
lc_model = XGBClassifier(
    scale_pos_weight=pos_weight,
    random_state=42,
    n_estimators=50,          # fewer trees = faster
    eval_metric='logloss',
    use_label_encoder=False
)

train_sizes, train_scores, val_scores = learning_curve(
    lc_model,
    X_train_final, y_train_final,
    cv=3,
    scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color='#D62728', lw=2, label='Training AUC')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#D62728')
ax.plot(train_sizes, val_mean, 's-', color='#1F77B4', lw=2, label='Cross-val AUC')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#1F77B4')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('AUC-ROC Score')
ax.set_title('Learning Curve\n(Does the model improve with more data?)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()

gap = train_mean[-1] - val_mean[-1]
print(f"Final training AUC   : {train_mean[-1]:.4f}")
print(f"Final cross-val AUC  : {val_mean[-1]:.4f}")
print(f"Overfit gap          : {gap:.4f}  {'(watch out)' if gap > 0.05 else '(acceptable)'}")

---
## Step 6: Final metrics summary

In [ ]:
from sklearn.metrics import classification_report

final_preds_optimal = (test_probs >= optimal_thresh).astype(int)

print("="*50)
print(" FULL MODEL SCORECARD (Test Split)")
print("="*50)
print(f"  Accuracy          : {accuracy_score(y_true_test, final_preds_optimal):.4f}")
print(f"  KS Statistic      : {ks_test:.4f}")
print(f"  Gini Coefficient  : {gini_test:.4f}")
print(f"  AUC-ROC           : {roc_auc_score(y_true_test, test_probs):.4f}")
print(f"  Optimal Threshold : {optimal_thresh:.2f}")
print(f"  Calib. Error(ECE) : {expected_calib_error:.4f}")
print("="*50)
print("\nClassification Report at optimal threshold:")
print(classification_report(y_true_test, final_preds_optimal, target_names=['Good','Bad']))

## Step 7: Predict on validation / test data

In [ ]:
production_model = final_model

val_raw = pd.read_csv(TEST_FILE)
print(f"Validation shape: {val_raw.shape}")
val_raw.head()

In [ ]:
# Encode categoricals
val_working = val_raw.copy()
val_cat_cols = val_working.select_dtypes(include=['object']).columns.tolist()
for col in val_cat_cols:
    val_working[col] = val_working[col].astype(str)
    val_working[col] = le.transform(val_working[col]) if col in cat_cols else le.fit_transform(val_working[col])

# Drop same columns as training
val_cols_to_remove = [c for c in val_working.columns if val_working[c].isnull().mean() > missing_cutoff]
val_cleaned = val_working.drop(columns=val_cols_to_remove, errors='ignore')

# Impute
val_imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
val_arr = val_cleaned.values
val_arr[:, ::] = val_imputer.fit_transform(val_arr[:, ::])
val_cleaned.iloc[:, ::] = val_arr[:, ::]

# Align columns to training features
for col in best_feats:
    if col not in val_cleaned.columns:
        val_cleaned[col] = 0   # fill any missing feature columns with 0

# Predict
risk_probabilities = production_model.predict_proba(val_cleaned[best_feats])

submission = pd.DataFrame()
submission[ID_COL]                  = val_cleaned[ID_COL]
submission['predicted_probability'] = risk_probabilities[:, 1]

submission.to_csv('output.csv', index=False)
print("Saved to output.csv")
submission.head(10)